# Ranking-algorithm effects

This notebook presents the 14 substantive orderings under the loose, unpinned interface. Estimates are discussion-level means from the completed inference tables; intervals are the prespecified paired bootstrap intervals. The explicit random ordering is the zero-calibration reference and is not ranked as a deployable policy.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent
INFERENCE_ROOT = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/inference'
REPORTING_ROOT = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/reporting'


## Run the relevant analysis if needed

This notebook is executable on its own once the upstream FORUM/ranking analysis handoff and policy scores exist. It runs paired bootstrap inference when its required result files are missing; otherwise it reads the frozen inference products.


In [ ]:
from commentgap_analysis.forum_scores import (
    DEFAULT_BOOTSTRAP_DRAWS, DEFAULT_RANDOM_DRAWS, DEFAULT_SEED,
    DEFAULT_TIE_DRAWS, run_policy_inference, run_policy_scoring
)

# Run the relevant analysis if outputs are missing or use the pre-reply/pin metric schema.
analysis_comments_path = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/analysis_comments.parquet'
policy_scores_path = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/policy_scores/policy_scores.parquet'
inference_targets = [INFERENCE_ROOT / 'policy_summary.csv', INFERENCE_ROOT / 'marginal_effects.csv', INFERENCE_ROOT / 'forum_ndcg_agreement.csv']
agreement_path = INFERENCE_ROOT / 'forum_ndcg_agreement.csv'
metric_schema_current = agreement_path.exists() and 'variant_group' in pd.read_csv(agreement_path, nrows=0).columns

if not all(path.exists() for path in inference_targets) or not metric_schema_current:
    prerequisites = [analysis_comments_path, policy_scores_path]
    missing_prerequisites = [str(path) for path in prerequisites if not path.exists()]
    if missing_prerequisites:
        raise FileNotFoundError(
            'Inference inputs are missing: ' + ', '.join(missing_prerequisites) +
            '. Run notebook 10 to build the upstream handoff and policy scores.'
        )
    print('Inference products are missing or stale; refreshing policy scores and paired bootstrap inference...')
    run_policy_scoring(
        analysis_path=analysis_comments_path,
        output_root=REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/policy_scores',
        tie_draws=DEFAULT_TIE_DRAWS, random_draws=DEFAULT_RANDOM_DRAWS, seed=DEFAULT_SEED,
        progress_every_stories=10,
    )
    run_policy_inference(
        policy_scores_path=policy_scores_path,
        analysis_comments_path=analysis_comments_path,
        output_root=INFERENCE_ROOT, bootstrap_draws=DEFAULT_BOOTSTRAP_DRAWS, seed=DEFAULT_SEED,
    )
else:
    print('Required grouped inference products already exist; reading them below.')


In [ ]:
summary = pd.read_csv(INFERENCE_ROOT / 'policy_summary.csv')
effects = pd.read_csv(INFERENCE_ROOT / 'marginal_effects.csv')
agreement = pd.read_csv(INFERENCE_ROOT / 'forum_ndcg_agreement.csv')
print(f"{effects['outcome'].nunique()} outcomes; {effects['contrast'].nunique()} contrasts; samples={effects['sample'].unique().tolist()}")


## Policy rankings across ordering, reply structure, and pinning

The raw table below reports average FORUM for the loose, unpinned policies, including random. The next table ranks all 84 substantive ordering × reply-mode × pinning bundles. Both top-10 and full-depth results are shown.


In [ ]:
for depth in ('top10', 'full'):
    print(f'Raw average FORUM: {depth}')
    raw_policy = summary[
        summary['sample'].eq('primary')
        & summary['depth'].eq(depth)
        & summary['reply_mode'].eq('loose')
        & ~summary['pinned']
    ].copy()
    raw_policy['interval'] = raw_policy.apply(lambda r: f"{r.estimate:.3f} [{r.ci_lower:.3f}, {r.ci_upper:.3f}]", axis=1)
    display(raw_policy[['ordering', 'estimate', 'ci_lower', 'ci_upper', 'probability_highest_forum', 'interval']].sort_values('estimate', ascending=False))


In [ ]:
for depth in ('top10', 'full'):
    ranked = summary[
        summary['sample'].eq('primary')
        & summary['depth'].eq(depth)
        & summary['deployable']
    ].copy()
    ranked['rank_within_outcome'] = ranked.groupby('outcome')['estimate'].rank(method='min', ascending=False).astype(int)
    print(f'Top substantive policy bundles by outcome: {depth}')
    display(ranked[ranked['rank_within_outcome'] <= 10].sort_values(['outcome', 'rank_within_outcome'])[[
        'outcome', 'rank_within_outcome', 'ordering', 'reply_mode', 'pinned',
        'estimate', 'ci_lower', 'ci_upper'
    ]])


In [ ]:
for depth in ('top10', 'full'):
    primary = effects[
        effects['sample'].eq('primary')
        & effects['depth'].eq(depth)
        & effects['contrast_family'].eq('ordering_vs_random')
    ].copy()
    primary['interval'] = primary.apply(lambda r: f"{r.estimate:.3f} [{r.ci_lower:.3f}, {r.ci_upper:.3f}]", axis=1)
    print(f'Ordering-versus-random contrasts: {depth}')
    display(primary.pivot(index='contrast', columns='outcome', values='interval'))


In [ ]:
import importlib
import commentgap_analysis.ranking_algorithm_effects as ranking_effects
importlib.reload(ranking_effects)
plot_ordering_effects = ranking_effects.plot_ordering_effects
plot_ordering_policy_variants = ranking_effects.plot_ordering_policy_variants

for depth in ('top10', 'full'):
    print(f'Ordering and average interface effects figure: {depth}')
    paths = plot_ordering_effects(effects, REPORTING_ROOT, depth=depth)
    figure_path = REPORTING_ROOT / ('figure_ordering_effects.png' if depth == 'top10' else f'figure_ordering_effects_{depth}.png')
    display(Image(filename=str(figure_path)))

    print(f'Ordering × reply × pin figure: {depth}')
    plot_ordering_policy_variants(summary, REPORTING_ROOT, depth=depth)
    figure_path = REPORTING_ROOT / ('figure_ordering_policy_variants.png' if depth == 'top10' else f'figure_ordering_policy_variants_{depth}.png')
    display(Image(filename=str(figure_path)))


## FORUM versus direct nDCG

nDCG is a standard discounted-cumulative-gain measure for ranked lists with binary or graded relevance (Järvelin & Kekäläinen, 2002, [doi:10.1145/582415.582418](https://doi.org/10.1145/582415.582418)). Here the continuous comment outcome is used as a graded gain after within-discussion min–max scaling, with linear rather than exponential gain. This is an adaptation of standard nDCG, not a new metric.


In [ ]:
from commentgap_analysis.ranking_algorithm_effects import plot_metric_agreement

for depth in ('top10', 'full'):
    agreement_view = agreement[
        agreement['sample'].eq('primary') & agreement['depth'].eq(depth)
    ].copy()
    print(f'FORUM/nDCG agreement: {depth}')
    display(agreement_view[['outcome', 'variant_group', 'reply_mode', 'pinned', 'n_stories', 'n_orderings', 'spearman_forum_ndcg', 'ci_lower', 'ci_upper']])
    plot_metric_agreement(agreement, REPORTING_ROOT, depth=depth)
    figure_path = REPORTING_ROOT / ('figure_forum_ndcg_agreement.png' if depth == 'top10' else f'figure_forum_ndcg_agreement_{depth}.png')
    display(Image(filename=str(figure_path)))
